# Phase 8 — Risque

## Pourquoi cette phase est importante

La phase 5 publie une volatilité et un drawdown maximal. La volatilité écarte chaque mois de la moyenne, qu'il soit favorable ou défavorable. Le drawdown suit la baisse depuis un sommet du placement. Aucun des deux ne dit quel mois appartient aux plus mauvais, ni si les titres de la liste bougent ensemble.

Cette phase ajoute deux lectures sur les mêmes mois, déjà publiés.

La première est la queue du rendement du portefeuille. La Value at Risk historique retient un rang dans les mois les plus mauvais. L'expected shortfall moyenne ces mois. Les trois livres sont comparés sur cette queue, pas seulement sur la volatilité.

La seconde est la dépendance entre titres. La corrélation de Pearson est calculée sur les titres inclus, sans leurs poids. La moyenne de ces paires dit si la liste, prise dans son ensemble, bouge ensemble. Elle peut rester modérée alors qu'un secteur, lui, est très lié. Le notebook d'analyse cherche ce bloc. La phase 6 reste le lieu des poids, du nombre effectif de titres et des plafonds.


## Conventions de lecture

Un lecteur vérifie d'abord l'unité, le signe et l'échantillon.

Le rendement est un rendement simple, en francs suisses, de fin de mois à fin de mois. Ce n'est pas un rendement logarithmique, et ce n'est pas un rendement journalier. Le poids du mois est celui déjà publié par la phase 5. Un rendement négatif est une perte.

\(T\) est le nombre de mois de la fenêtre des portefeuilles. Il est lu dans le fichier. Les trois livres partagent ces mois et les titres inclus. Un livre est l'un des trois portefeuilles de la phase 5 : équipondéré, volatilité minimale, Sharpe maximal. D'un livre à l'autre, seuls les poids changent.

Une règle est lue dans le module ou dans l'IPS : `VAR_LEVEL`, les seuils de lecture, les plafonds de poids. Une mesure est calculée sur les rendements : volatilité, VaR, expected shortfall, corrélation. La note imprime la mesure, puis elle la recalcule. Elle s'arrête si les deux s'écartent de plus de \(10^{-8}\).


## Qui réécrit la page

La phrase en tête de `reports/risk/dashboard.html` n'est pas saisie à la main. `python -m src.pipeline.refresh` appelle `build_risk`. Cette fonction écrit les fichiers de `data/07_risk/`, puis appelle `write_dashboard`.

Le texte modèle est dans `src/risk/dashboard.py`. Au moment de l'écriture, le programme y place la période et le nombre de titres lus dans les fichiers qu'il vient d'écrire, ainsi que la corrélation moyenne. Le seuil affiché est `1 - VAR_LEVEL`, et la part de queue est `VAR_LEVEL`, la même constante que le calcul. Le reste de la phrase décrit la règle. Il ne dépend pas des cours du jour.

Le fichier HTML conserve la dernière reconstruction. Tant que `build_risk` n'a pas été relancé, la phrase affiche les chiffres de cette reconstruction. Le même rafraîchissement réécrit la phrase et les graphiques.


## Où en est le calcul

Le module est `src/risk/build_risk.py`. La commande directe est `python -m src.risk.build_risk`. Elle fait le même calcul que l'étape risque du rafraîchissement.

La volatilité recalculée sur `portfolio_return` doit retrouver `annualized_volatility` de `portfolio_metrics.csv`, à \(10^{-8}\) près. Sinon le calcul s'arrête. La VaR n'est pas recalculée par les notebooks : ils lisent `risk_by_strategy.csv` et vérifient que le rang publié est bien celui de la formule.

`notebooks/08_analyse_risque.ipynb` lit la matrice, les secteurs et les poids. Il ne change ni la VaR, ni les poids. `reports/risk/08_rapport_risque.ipynb` imprime les chiffres de la reconstruction.


## Sources lues

Le module lit le rendement du portefeuille et les cours des titres. Le notebook d'analyse lit en plus le classement et les poids déjà publiés par les phases 5 et 6.

| Fichier | Qui le lit | Usage |
|---|---|---|
| `data/04_portfolios/portfolio_monthly_returns.csv` | le module | Volatilité, VaR, expected shortfall, sur `portfolio_return` |
| `data/04_portfolios/portfolio_metrics.csv` | le module | Contrôle de la volatilité |
| `data/04_portfolios/portfolio_membership.csv` | le module | Les titres dont `decision` vaut `included` |
| `data/02_processed/market/market_prices_chf.csv` | le module | Dernier cours CHF du mois, pour la corrélation |
| `data/02_processed/macro/saron.csv` | la phase 5, en amont | La période scorée est déjà celle du portefeuille. Ce module ne rouvre pas le calendrier |
| Classement des positions du fonds | le notebook d'analyse | Secteur joint par ticker, la même fonction que la phase 6 |
| `data/04_portfolios/portfolio_weights.csv` | le notebook d'analyse | Poids de chaque ligne au mois de rapport |
| `data/05_monitoring/allocation_by_sector.csv` | le notebook d'analyse | Poids de secteur au mois de rapport |
| `data/05_monitoring/concentration.csv` | le notebook d'analyse | Nombre effectif de titres |
| `data/05_monitoring/monthly_report.csv` | le notebook d'analyse | Mois de rapport, qui doit être la fin de fenêtre |

`DS-006` n'est pas lu. Le statut a déjà choisi les titres en phase 5.

`build_risk` n'ouvre pas le fichier de positions du fonds. Le notebook d'analyse, lui, appelle `load_holding_classification`, comme la phase 6, pour attacher un secteur à chaque ticker inclus. Sans ce joint, la matrice resterait un panneau de symboles.


## Volatilité

Même définition que la phase 5. $s$ est l'écart-type d'échantillon des rendements du portefeuille, diviseur $T - 1$.

$$
\sigma_p^{\mathrm{ann}} = s(R_p) \times \sqrt{12}
$$

Ce nombre décrit l'ensemble des mois. Il ne désigne pas les mois de la queue. C'est pourquoi un livre peut avoir la volatilité la plus basse et une VaR plus profonde qu'un autre : la volatilité est tirée par tous les écarts, la VaR par un seul rang. Le notebook d'analyse imprime les deux classements à partir du fichier, sans supposer lequel l'emporte.


## Corrélation

On prend les titres dont `decision` vaut `included`. Leur rendement mensuel est celui de la phase 5 : dernier cours CHF positif du mois, divisé par le dernier cours CHF positif du mois précédent, moins un. Les mois sont ceux de `portfolio_monthly_returns.csv`.

La corrélation publiée est celle de Pearson, avec le même diviseur $T - 1$ que la volatilité. $s_i$ est l'écart-type d'échantillon du titre $i$.

$$
\rho_{ij} = \frac{\sum_{t=1}^{T}(r_{i,t}-\bar r_i)(r_{j,t}-\bar r_j)}{(T-1)\, s_i s_j}
$$

Elle vaut 1 pour un titre avec lui-même. Une série constante n'arrive pas jusqu'ici : le module s'arrête. Chaque mois pèse autant que les autres. Ce n'est pas une corrélation calculée sur les seuls mois de queue.

Le résumé n'est pas la matrice. C'est la moyenne des corrélations hors diagonale, une fois par paire :

$$
\bar\rho = \frac{2}{N(N-1)} \sum_{i < j} \rho_{ij}
$$

$N$ est le nombre de titres inclus. Cette moyenne décrit la liste. Elle est la même pour les trois stratégies, parce qu'aucun poids n'entre dans la formule. Deux titres comptent une fois, que le livre les tienne au plafond ou pas du tout.

Une moyenne modérée n'empêche pas un secteur d'être très lié. Le notebook d'analyse regroupe donc les paires par secteur et retient celles qui dépassent deux seuils de lecture, `PAIR_LIST_FLOOR` et `PAIR_LIST_TAIL`, définis dans le module. Ce ne sont pas des plafonds de l'IPS. La cellule de cette note imprime leur valeur. Un secteur n'est appelé le plus lié que s'il a au moins deux paires : une seule paire ne fait pas un bloc.


## Value at Risk historique et expected shortfall

La VaR retenue est historique. Aucune loi normale n'est supposée. On ne publie pas une VaR à un jour, ni une VaR réglementaire, ni une prévision du mois suivant.

$\alpha$ est `VAR_LEVEL`. On trie les $T$ rendements mensuels du portefeuille du plus mauvais au meilleur. Si deux mois ont le même rendement, le mois le plus ancien vient d'abord. Le rang, en comptant à partir de 1, est

$$
k = \lceil \alpha \times T \rceil
$$

et $k$ vaut au moins 1. La VaR est le rendement de ce rang : le moins mauvais des $k$ mois les plus mauvais. L'expected shortfall est la moyenne de ces $k$ rendements. Il est donc inférieur ou égal à la VaR. Le module s'arrête si ce n'est pas le cas.

La lecture d'une VaR négative est une perte sur le mois, dans cette queue déjà observée. Si le rendement retenu vaut $-0{,}08$, la lecture est une perte de 8 % sur le mois. Cet exemple fixe l'unité. Ce n'est pas le chiffre du portefeuille : le chiffre est celui du fichier.

L'écart entre l'expected shortfall et la VaR dit si les mois pires que le seuil sont proches de ce seuil, ou nettement plus mauvais. Le notebook d'analyse imprime cet écart pour chaque livre.

Les mois du groupe sont écrits dans `var_tail.csv`, avec leur rang. Un mois présent dans les trois livres est une date commune de la liste. Un mois présent dans un seul livre tient à la règle de poids de ce livre.


## Ce que le notebook d'analyse ajoute

Le module s'arrête à la matrice et à la moyenne. Le notebook pose trois questions, et il y répond avec les fichiers de l'exécution.

La volatilité, la VaR et l'expected shortfall classent-ils les livres dans le même ordre ?

La moyenne cache-t-elle un secteur ? Le secteur vient du classement de la phase 6. La matrice dessinée est carrée, ordonnée par secteur puis par nom. L'axe porte le ticker, pour que les cases restent lisibles. Le survol donne le nom et le secteur. Le rouge est une corrélation haute. La phrase sous le graphique nomme le secteur le plus lié déjà calculé.

Ce secteur, et les paires au-dessus du premier seuil, pèsent-ils au mois de rapport ? Le poids lu est celui de `portfolio_weights.csv` à la fin de la fenêtre. Une paire très liée et de poids nul est une dépendance de la liste. Une paire très liée et de poids positif est une dépendance à l'intérieur du livre. Le plafond par émetteur est celui de l'IPS. Lorsqu'il est cité, la cellule imprime la valeur lue dans la configuration.


## Limites

La fenêtre est celle des portefeuilles. Un mois nouveau n'entre dans la VaR que lorsque cette fenêtre avance. Le rang $k$ peut alors changer, et la VaR avec lui. C'est le fonctionnement prévu, pas un chiffre figé.

La corrélation donne le même poids à chaque mois. Un mois de forte baisse est un point parmi $T$. Elle ne mesure pas à elle seule la dépendance dans la queue.

La matrice n'est pas multipliée par les poids. La volatilité du livre, elle, vient de `portfolio_return`, qui contient déjà les poids de chaque mois. On ne republie pas une volatilité reconstruite par $w^{\top}\Sigma w$.

La page HTML sur le disque est la dernière reconstruction. Elle se met à jour quand `build_risk` tourne, pas quand on l'ouvre.


## Fichiers publiés

`data/07_risk/risk_by_strategy.csv`, une ligne par stratégie : nombre de mois, fenêtre, volatilité annualisée, VaR, expected shortfall.

`data/07_risk/var_tail.csv` : stratégie, rang, mois, rendement. Les rangs vont de 1 à $k$. Leur moyenne est l'expected shortfall. Le dernier est la VaR.

`data/07_risk/universe_correlation.csv`, une ligne : nombre de titres inclus, corrélation moyenne hors diagonale, nombre de mois.

`data/07_risk/correlation_matrix.csv` : la matrice de Pearson. La première colonne est le symbole. La diagonale vaut 1.

La cellule suivante relit ces fichiers. Elle recalcule le rang, la VaR, l'expected shortfall et la moyenne des paires, et elle s'arrête si le fichier s'en écarte.


In [1]:
import math
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "src").is_dir():
        PROJECT_ROOT = candidate
        break
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src.portfolio_construction.build_portfolios import MONTHLY_RETURNS_FILE
from src.risk.build_risk import (
    AGREEMENT,
    MATRIX_FILE,
    METRICS_OUTPUT,
    PAIR_LIST_FLOOR,
    PAIR_LIST_TAIL,
    SUMMARY_FILE,
    TAIL_FILE,
    VAR_LEVEL,
)

LABELS = {
    "equal_weight": "Équipondéré",
    "minimum_volatility": "Volatilité minimale",
    "maximum_sharpe": "Sharpe maximal",
}


def percent(value):
    return f"{float(value) * 100:.1f}".replace(".", ",") + " %"


print(
    f"Seuil lu dans le module : {percent(1 - VAR_LEVEL)}. "
    f"Part de queue : {percent(VAR_LEVEL)}."
)
print(
    f"Seuils de lecture des paires : {percent(PAIR_LIST_FLOOR)} et {percent(PAIR_LIST_TAIL)}."
)

metrics = pd.read_csv(METRICS_OUTPUT)
tail = pd.read_csv(TAIL_FILE)
summary = pd.read_csv(SUMMARY_FILE)
returns = pd.read_csv(MONTHLY_RETURNS_FILE)
returns["month"] = returns["month"].astype(str)
for strategy, label in LABELS.items():
    block = returns.loc[returns["strategy"].eq(strategy)].sort_values(
        ["portfolio_return", "month"],
        ascending=[True, True],
        kind="mergesort",
    )
    rank = math.ceil(VAR_LEVEL * len(block))
    recomputed_var = float(block["portfolio_return"].iloc[rank - 1])
    recomputed_shortfall = float(block["portfolio_return"].iloc[:rank].mean())
    published = metrics.loc[metrics["strategy"].eq(strategy)].iloc[0]
    if abs(recomputed_var - float(published["var_95"])) > AGREEMENT:
        raise RuntimeError(f"{strategy} VaR does not match the file.")
    if abs(recomputed_shortfall - float(published["expected_shortfall_95"])) > AGREEMENT:
        raise RuntimeError(f"{strategy} expected shortfall does not match the file.")
    if len(tail.loc[tail["strategy"].eq(strategy)]) != rank:
        raise RuntimeError(f"{strategy} tail does not have {rank} months.")
    gap = float(published["var_95"]) - float(published["expected_shortfall_95"])
    print(
        f"{label} : rang {rank}, VaR {percent(published['var_95'])}, "
        f"expected shortfall {percent(published['expected_shortfall_95'])}, "
        f"plus profond de {percent(gap)}. Le fichier est retrouvé."
    )

matrix = pd.read_csv(MATRIX_FILE, index_col="symbol")
values = matrix.to_numpy(dtype=float)
pairs = values[np.triu_indices(values.shape[0], k=1)]
average = float(pairs.mean())
if abs(average - float(summary["average_correlation"].iloc[0])) > AGREEMENT:
    raise RuntimeError("The average correlation does not match the file.")
series = pd.Series(pairs)
print(
    f"Paires : {len(series)}. Minimum {percent(series.min())}. "
    f"Premier quartile {percent(series.quantile(0.25))}. "
    f"Médiane {percent(series.median())}. "
    f"Troisième quartile {percent(series.quantile(0.75))}. "
    f"Maximum {percent(series.max())}."
)
print(
    f"Paires d'au moins {percent(PAIR_LIST_FLOOR)} : {int((series >= PAIR_LIST_FLOOR).sum())}. "
    f"D'au moins {percent(PAIR_LIST_TAIL)} : {int((series >= PAIR_LIST_TAIL).sum())}."
)


Seuil lu dans le module : 95,0 %. Part de queue : 5,0 %.
Seuils de lecture des paires : 50,0 % et 70,0 %.
Équipondéré : rang 6, VaR -5,5 %, expected shortfall -8,0 %, plus profond de 2,5 %. Le fichier est retrouvé.
Volatilité minimale : rang 6, VaR -6,2 %, expected shortfall -7,6 %, plus profond de 1,4 %. Le fichier est retrouvé.
Sharpe maximal : rang 6, VaR -6,5 %, expected shortfall -8,6 %, plus profond de 2,1 %. Le fichier est retrouvé.
Paires : 406. Minimum -14,9 %. Premier quartile 6,3 %. Médiane 15,1 %. Troisième quartile 25,4 %. Maximum 77,2 %.
Paires d'au moins 50,0 % : 13. D'au moins 70,0 % : 4.


# Dictionnaire des termes de cette note

Ce dictionnaire reprend les termes employés dans cette note. Il n'en ajoute pas. Les formules restent dans la section qui les écrit.

**Corrélation de Pearson.** Lien linéaire entre les rendements mensuels de deux titres sur les mois communs. Elle vaut 1 pour un titre avec lui-même. Chaque mois a le même poids.

**Corrélation moyenne.** Moyenne des corrélations hors diagonale, une fois par paire. Elle décrit la liste partagée, pas les poids d'une stratégie.

**Drawdown maximal.** Plus forte baisse déjà publiée par la phase 5. Cette note ne la recalcule pas.

**Expected shortfall.** Moyenne des rendements mensuels du rang 1 au rang $k$. Il est inférieur ou égal à la VaR du même seuil.

**Livre.** L'un des trois portefeuilles de la phase 5. Les trois partagent les titres et les mois. Seuls les poids diffèrent.

**Matrice de corrélation.** Tableau de Pearson des titres inclus. La diagonale vaut 1. Dans le graphique du notebook d'analyse, l'axe porte le ticker.

**Queue.** Les $k$ mois les plus mauvais du portefeuille.

**Rang $k$.** $\lceil \alpha \times T \rceil$, avec $\alpha$ égal à `VAR_LEVEL`. La VaR est le rendement de ce rang.

**SARON.** La période scorée est celle des portefeuilles, déjà alignée sur le SARON et sur le dernier mois publié par le fonds.

**Seuil de lecture.** `PAIR_LIST_FLOOR` ou `PAIR_LIST_TAIL`. Sert à lister des paires. Ce n'est pas un plafond de l'IPS.

**Value at Risk historique (VaR).** Rendement du rang $k$, sans loi normale. Ce n'est pas une prévision du mois suivant.

**Volatilité.** Écart-type d'échantillon des rendements du portefeuille, annualisé avec la racine de 12. Même définition que la phase 5.
